# 🎙️ AI Music OS — Kokoro TTS V3

Kokoro Text-to-Speech for Google Colab.

### V3 fixes
- Python 3.13 compatible installation path
- Official Kokoro GitHub source on Python 3.13+
- Separate dependency installation
- Google Drive persistent Hugging Face cache
- Google Drive output storage
- Real Kokoro generation only
- No fake/sine-wave fallback
- Real generation smoke test before Gradio
- Gradio web interface


In [ ]:
# ============================================================
# AI MUSIC OS — KOKORO TTS V3
# ============================================================

import os
import sys
import subprocess
import time
from pathlib import Path

print('=' * 70)
print('AI MUSIC OS — KOKORO TTS V3')
print('=' * 70)
print('Python:', sys.version)
print('Executable:', sys.executable)


# ============================================================
# GOOGLE DRIVE
# ============================================================

from google.colab import drive
        
print('Mounting Google Drive...')
drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/AI_Music_OS')
CACHE_DIR = ROOT / 'cache' / 'huggingface'
OUTPUT_DIR = ROOT / 'outputs' / 'kokoro'
LOG_DIR = ROOT / 'logs'

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(CACHE_DIR)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE_DIR / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(CACHE_DIR / 'transformers')

print('Root:', ROOT)
print('HF cache:', CACHE_DIR)
print('Output:', OUTPUT_DIR)
        

# ============================================================
# HELPER — PIP INSTALL
# ============================================================

def run_pip(args, name):
    command = [sys.executable, '-m', 'pip'] + args
    print('\n' + '=' * 70)
    print(name)
    print('=' * 70)
    print('COMMAND:', ' '.join(command))

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    print(result.stdout)

    if result.returncode != 0:
        raise RuntimeError(
            name + ' FAILED. Exit code: ' + str(result.returncode)
        )

    print(name + ': OK')


# ============================================================
# ESPEAK-NG
# ============================================================

result = subprocess.run(
    ['apt-get', '-qq', '-y', 'install', 'espeak-ng'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print(result.stdout[-5000:])

if result.returncode != 0:
    raise RuntimeError('espeak-ng installation failed.')

print('espeak-ng: OK')


# ============================================================
# PYTHON PACKAGES
# ============================================================

run_pip(
    ['install', '-U', 'pip', 'setuptools', 'wheel'],
    'Upgrade pip tools'
)

if sys.version_info >= (3, 13):
    print('\nPython 3.13+ detected.')
    print('Using current upstream Misaki and Kokoro sources.')

    run_pip(
        [
            'install', '-U',
            'numpy',
            'num2words',
            'spacy',
            'loguru'
        ],
        'Install Misaki runtime dependencies'
    )

    run_pip(
        [
            'install', '-U', '--no-cache-dir',
            'git+https://github.com/hexgrad/misaki.git'
        ],
        'Install Misaki from official GitHub'
    )

    run_pip(
        [
            'install', '-U', '--no-cache-dir', '--no-deps',
            'git+https://github.com/hexgrad/kokoro.git'
        ],
        'Install Kokoro from official GitHub'
    )

else:
    print('\nPython <= 3.12 detected.')
        
    run_pip(
        ['install', '-U', 'misaki[en]>=0.9.4'],
        'Install Misaki'
    )

    run_pip(
        ['install', '-U', 'kokoro>=0.9.4'],
        'Install Kokoro'
    )


run_pip(
    [
        'install', '-U',
        'huggingface_hub',
        'transformers',
        'soundfile',
        'numpy'
    ],
    'Install Kokoro runtime dependencies'
)

run_pip(
    ['install', '-U', 'gradio>=5,<7'],
    'Install Gradio'
)


# ============================================================
# VERIFY IMPORTS
# ============================================================

print('\n' + '=' * 70)
print('VERIFYING KOKORO')
print('=' * 70)

import torch
import numpy as np
import soundfile as sf

try:
    import kokoro
    from kokoro import KPipeline
except Exception as e:
    print('Kokoro import failed:')
    print(repr(e))
    raise

print('Kokoro module:', kokoro.__file__)
print('PyTorch:', torch.__version__)
print('NumPy:', np.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU: NOT AVAILABLE')
    print('Kokoro will use CPU.')


# ============================================================
# LOAD PIPELINE
# ============================================================

print('\n' + '=' * 70)
print('LOADING KOKORO PIPELINE')
print('=' * 70)

pipeline = KPipeline(lang_code='a')

print('Kokoro pipeline loaded successfully.')


# ============================================================
# VOICES
# ============================================================

VOICE_CHOICES = [
    'af_heart',
    'af_bella',
    'af_nicole',
    'af_sarah',
    'af_sky',
    'am_adam',
    'am_michael',
    'bf_emma',
    'bf_isabella',
    'bm_george',
    'bm_lewis'
]

print('\nAvailable voices:')
for voice in VOICE_CHOICES:
    print(' -', voice)


# ============================================================
# GENERATION FUNCTION
# ============================================================

def generate_audio(text, voice='af_heart', speed=1.0, output_name=None):

    text = (text or '').strip()

    if not text:
        raise ValueError('Text is empty.')

    if voice not in VOICE_CHOICES:
        raise ValueError(
            'Unknown voice: ' + str(voice) +
            '\nAvailable voices: ' + str(VOICE_CHOICES)
        )

    speed = float(speed)

    if speed <= 0:
        raise ValueError('Speed must be greater than 0.')

    print('\n' + '-' * 70)
    print('Generating real Kokoro audio')
    print('-' * 70)
    print('Voice:', voice)
    print('Speed:', speed)
    print('Text:', text[:300])

    chunks = []

    generator = pipeline(
        text,
        voice=voice,
        speed=speed
    )

    for index, item in enumerate(generator):

        if not isinstance(item, tuple) or len(item) < 3:
            raise RuntimeError(
                'Unexpected Kokoro output format: ' + repr(item)[:1000]
            )

        graphemes, phonemes, audio = item

        print(
            'Chunk', index + 1,
            '| graphemes:', len(graphemes),
            '| phonemes:', len(phonemes)
        )

        if audio is None:
            raise RuntimeError(
                'Kokoro returned empty audio.'
            )

        chunks.append(
            np.asarray(audio, dtype=np.float32)
        )

    if not chunks:
        raise RuntimeError('Kokoro returned no audio chunks.')

    audio = np.concatenate(chunks)

    if audio.size == 0:
        raise RuntimeError('Generated audio is empty.')

    sample_rate = 24000

    if output_name:
        filename = str(output_name)
        if not filename.lower().endswith('.wav'):
            filename += '.wav'
    else:
        filename = 'kokoro_' + str(int(time.time())) + '.wav'

    output_path = OUTPUT_DIR / filename

    sf.write(
        str(output_path),
        audio,
        sample_rate
    )

    if not output_path.exists():
        raise RuntimeError('Audio file was not created.')

    size = output_path.stat().st_size

    if size < 1000:
        raise RuntimeError('Generated WAV file is suspiciously small.')

    duration = len(audio) / sample_rate

    print('\nSUCCESS')
    print('File:', output_path)
    print('Size:', size, 'bytes')
    print('Sample rate:', sample_rate)
    print('Duration:', round(duration, 2), 'seconds')

    return str(output_path)


# ============================================================
# REAL SMOKE TEST
# ============================================================

print('\n' + '=' * 70)
print('RUNNING REAL KOKORO SMOKE TEST')
print('=' * 70)

TEST_TEXT = (
    'Hello. This is a real Kokoro text to speech test. '
    'The audio is generated by the Kokoro model. '
    'This confirms that the model and audio output are working correctly.'
)

TEST_FILE = generate_audio(
    TEST_TEXT,
    voice='af_heart',
    speed=1.0,
    output_name='kokoro_smoke_test.wav'
)

from IPython.display import Audio, display

print('\nPlaying smoke test audio...')
display(Audio(filename=TEST_FILE))


# ============================================================
# GRADIO UI
# ============================================================

import gradio as gr


def ui_generate(text, voice, speed):
    try:
        return generate_audio(
            text=text,
            voice=voice,
            speed=speed
        )
    except Exception as e:
        print('\nGENERATION ERROR:')
        print(repr(e))
        raise gr.Error(str(e))


demo = gr.Interface(
    fn=ui_generate,
    inputs=[
        gr.Textbox(
            lines=8,
            label='Text',
            placeholder='Type the text you want Kokoro to speak...',
            value='Hello. Welcome to the AI Music OS Kokoro voice generator.'
        ),
        gr.Dropdown(
            choices=VOICE_CHOICES,
            value='af_heart',
            label='Voice'
        ),
        gr.Slider(
            minimum=0.5,
            maximum=2.0,
            value=1.0,
            step=0.05,
            label='Speech Speed'
        )
    ],
        
    outputs=gr.Audio(
        label='Generated Audio',
        type='filepath'
    ),
    title='AI Music OS — Kokoro TTS V3',
    description='Kokoro Text-to-Speech with Google Drive persistent storage.'
)

print('\n' + '=' * 70)
print('KOKORO UI READY')
print('=' * 70)
print('Launching Gradio...')

demo.launch(
    share=True,
    debug=True,
    show_error=True
)
